Step 10 — Eventually we'll create the Employee 360
Once we know your five tables, the notebook will become something like:

                    PostgreSQL
                         │
             ┌───────────┼───────────┐
             │           │           │
             ▼           ▼           ▼
        Employees    Salary      Performance
             │
             ├──────── Departments
             │
             └──────── Promotions
                         │
                         ▼
                  Data Cleaning
                         │
                         ▼
                     Joins
                         │
                         ▼
                Business Metrics
                         │
                         ▼
                  Employee 360
                         │
                         ▼
              output/employee_360/
                         │
                         ▼
                      Parquet

And we'll calculate:

Total compensation
Salary range status
Salary validation
Employee age
Years of service
Promotion status
Department salary analysis
Performance analysis

Let's do this incrementally
Don't try to run the entire pipeline yet.

First get these three things working:

✅ Fresh hr_employee_360.ipynb
✅ PySpark starts successfully
✅ PostgreSQL connection works through JDBC
Then we'll build the actual Employee 360 pipeline.

In [1]:
print("HR Employee 360 project started!")

HR Employee 360 project started!


Start Spark

In [9]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("HR Employee 360")
    .config(
        "spark.jars.packages",
        "org.postgresql:postgresql:42.7.8"
    )
    .getOrCreate()
)

print("Spark version:", spark.version)
print("PostgreSQL JDBC package:",
      spark.sparkContext.getConf().get("spark.jars.packages"))


Spark version: 4.2.0
PostgreSQL JDBC package: org.postgresql:postgresql:42.7.8


Test whether Java can now find the PostgreSQL driver

In [10]:
print(
    spark.sparkContext.getConf().get(
        "spark.jars.packages"
    )
)



org.postgresql:postgresql:42.7.8


Load database configuration

In [11]:
import sys
import os

sys.path.append(os.path.abspath(".."))

from config.database_config import get_database_config

config = get_database_config()

print("Database:", config["database"])
print("Host:", config["host"])
print("Port:", config["port"])
print("User:", config["user"])
print("JDBC URL:", config["jdbc_url"])


Database: HR
Host: localhost
Port: 5432
User: vikas
JDBC URL: jdbc:postgresql://localhost:5432/HR


Create a reusable PostgreSQL reader
Instead of repeating the JDBC code five times, create a function

In [5]:
def read_postgres_table(table_name):
    return (
        spark.read
        .format("jdbc")
        .option("url", config["jdbc_url"])
        .option("dbtable", table_name)
        .option(
            "user",
            config["connection_properties"]["user"]
        )
        .option(
            "password",
            config["connection_properties"]["password"]
        )
        .option(
            "driver",
            config["connection_properties"]["driver"]
        )
        .load()
    )


In [14]:
employees = read_postgres_table("hr_data.employees")

Extract all five tables

In [15]:
employees = read_postgres_table("hr_data.employees")

departments = read_postgres_table("hr_data.departments")

salary = read_postgres_table("hr_data.salary")

job_roles = read_postgres_table("hr_data.job_roles")

performance = read_postgres_table("hr_data.performance")


In [20]:
print("-------employees--------")
employees.show(5)
print("-------departments--------")
departments.show(5)
print("-------salary--------")
salary.show(5)
print("-------performance--------")
performance.show(5)
print("-------job_roles--------")
job_roles.show(5)


-------employees--------
+-----------+----------+---------+------+----------+-------------+------+----------+----------+---------------+------+
|employee_id|first_name|last_name|gender|       dob|department_id|job_id|manager_id| hire_date|employment_type|status|
+-----------+----------+---------+------+----------+-------------+------+----------+----------+---------------+------+
|       E001|     Aarav|   Sharma|  Male|1995-03-15|          D01|   J01|      E010|2020-05-12|      Full-Time|Active|
|       E002|     Priya|    Patel|Female|1992-08-21|          D02|   J02|      E011|2019-02-15|      Full-Time|Active|
|       E003|     Rahul|    Verma|  Male|1988-01-10|          D03|   J03|      E012|2016-06-20|      Full-Time|Active|
|       E004|      Neha|    Singh|Female|1997-02-14|          D04|   J04|      E013|2022-07-18|      Full-Time|Active|
|       E005|     Rohan|    Mehta|  Male|1994-08-01|          D05|   J05|      E014|2019-09-01|      Full-Time|Active|
+-----------+----------

Check the number of records:

In [22]:
print("Employees:", employees.count())
print("Departments:", departments.count())
print("Salaries:", salary.count())
print("Performance:", performance.count())
print("job_roles:", job_roles.count())


Employees: 50
Departments: 6
Salaries: 50
Performance: 50
job_roles: 8


Check schemas

In [23]:
employees.printSchema()
departments.printSchema()
salary.printSchema()
performance.printSchema()
job_roles.printSchema()


root
 |-- employee_id: string (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- dob: date (nullable = true)
 |-- department_id: string (nullable = true)
 |-- job_id: string (nullable = true)
 |-- manager_id: string (nullable = true)
 |-- hire_date: date (nullable = true)
 |-- employment_type: string (nullable = true)
 |-- status: string (nullable = true)

root
 |-- department_id: string (nullable = true)
 |-- department_name: string (nullable = true)
 |-- location: string (nullable = true)
 |-- head_of_department: string (nullable = true)

root
 |-- employee_id: string (nullable = true)
 |-- basic_salary: decimal(12,2) (nullable = true)
 |-- bonus: decimal(12,2) (nullable = true)
 |-- allowance: decimal(12,2) (nullable = true)
 |-- deductions: decimal(12,2) (nullable = true)
 |-- net_salary: decimal(12,2) (nullable = true)

root
 |-- performance_id: string (nullable = true)
 |-- employee_id: s

Data Quality Checks

In [24]:
employees.groupBy("employee_id") \
    .count() \
    .filter("count > 1") \
    .show()


+-----------+-----+
|employee_id|count|
+-----------+-----+
+-----------+-----+



Null employee IDs

In [25]:
employees.filter(
    employees.employee_id.isNull()
).show()


+-----------+----------+---------+------+---+-------------+------+----------+---------+---------------+------+
|employee_id|first_name|last_name|gender|dob|department_id|job_id|manager_id|hire_date|employment_type|status|
+-----------+----------+---------+------+---+-------------+------+----------+---------+---------------+------+
+-----------+----------+---------+------+---+-------------+------+----------+---------+---------------+------+



Null names

In [26]:
employees.filter(
    employees.first_name.isNull() |
    employees.last_name.isNull()
).show()


+-----------+----------+---------+------+---+-------------+------+----------+---------+---------------+------+
|employee_id|first_name|last_name|gender|dob|department_id|job_id|manager_id|hire_date|employment_type|status|
+-----------+----------+---------+------+---+-------------+------+----------+---------+---------------+------+
+-----------+----------+---------+------+---+-------------+------+----------+---------+---------------+------+

